In [10]:
import gym
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical

# 비트코인 거래 환경 정의
class TradingEnv(gym.Env):
    def __init__(self, data):
        super(TradingEnv, self).__init__()
        self.data = data
        self.current_step = 0
        self.action_space = gym.spaces.Discrete(3)  # 0: 매도, 1: 관망, 2: 매수
        self.observation_space = gym.spaces.Box(low=-np.inf, high=np.inf, shape=(len(data.columns),), dtype=np.float32)
        self.balance = 10000  # 초기 자본
        self.holdings = 0

    def reset(self):
        self.current_step = 0
        self.balance = 10000
        self.holdings = 0
        return self.data.iloc[self.current_step].values

    def step(self, action):
        self.current_step += 1
        done = self.current_step >= len(self.data) - 1
        reward = 0

        price = self.data.iloc[self.current_step]['Close']
        if action == 2 and self.balance > 0:  # 매수
            self.holdings += self.balance/price
            self.balance -= self.holdings*price
            print(f'balance:{self.balance}, holdings:{self.holdings}')
        elif action == 0 and self.holdings > 0:  # 매도
            self.balance += self.holdings*price
            reward += self.holdings*price  # 이익 반영
            self.holdings = 0
            print(f'balance:{self.balance}, holdings:{self.holdings}, reward:{reward}')

        next_state = self.data.iloc[self.current_step].values
        return next_state, reward, done, {}

# 신경망 정의
class PPOModel(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(PPOModel, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU()
        )
        self.policy = nn.Linear(64, output_dim)
        self.value = nn.Linear(64, 1)

    def forward(self, x):
        x = self.fc(x)
        policy = self.policy(x)
        value = self.value(x)
        return policy, value

# PPO 에이전트 정의
class PPOAgent:
    def __init__(self, state_dim, action_dim, lr=0.001, gamma=0.99, eps_clip=0.2):
        self.model = PPOModel(state_dim, action_dim)
        self.optimizer = optim.Adam(self.model.parameters(), lr=lr)
        self.gamma = gamma
        self.eps_clip = eps_clip
        self.policy_old = PPOModel(state_dim, action_dim)
        self.policy_old.load_state_dict(self.model.state_dict())

    def select_action(self, state):
        state = torch.FloatTensor(state).unsqueeze(0)
        logits, _ = self.model(state)
        probs = Categorical(logits=logits)
        action = probs.sample()
        
        return action.item(), probs.log_prob(action)

    def update(self, memory):
        states, actions, rewards, log_probs = zip(*memory)
        states = torch.FloatTensor(states)
        actions = torch.tensor(actions)
        rewards = torch.tensor(rewards)
        old_log_probs = torch.stack(log_probs)

        for _ in range(5):  # 여러 번 업데이트
            logits, state_values = self.model(states)
            new_probs = Categorical(logits=logits)
            new_log_probs = new_probs.log_prob(actions)
            ratios = torch.exp(new_log_probs - old_log_probs)
            advantages = rewards - state_values.detach().squeeze()
            surr1 = ratios * advantages
            surr2 = torch.clamp(ratios, 1 - self.eps_clip, 1 + self.eps_clip) * advantages
            loss = torch.tensor(-torch.min(surr1, surr2).mean() + nn.MSELoss()(state_values.squeeze(), rewards), requires_grad=True, dtype=torch.float32)
            loss.requires_grad_(True)
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
        
        self.policy_old.load_state_dict(self.model.state_dict())

# 데이터 로드 및 환경 설정
data = pd.read_csv(f'/workspace/data/raw/BTCUSDT/BTCUSDT-1h-2021.csv', index_col=0)
data = data[['Open','High','Low','Close']]

env = TradingEnv(data)
agent = PPOAgent(state_dim=len(data.columns), action_dim=3)

# 학습 과정
episodes = 5
gamma = 0.99
for episode in range(episodes):
    state = env.reset()
    done = False
    memory = []
    total=0
    while not done:
        action, log_prob = agent.select_action(state)
        if action == 0:
            print(0)
        next_state, reward, done, _ = env.step(action)
        memory.append((state, action, reward, log_prob))
        state = next_state
        total += reward
    print(f'episode:{episode}, total:{total}')
    agent.update(memory)

# 테스트 실행
state = env.reset()
done = False
while not done:
    action, _ = agent.select_action(state)
    state, reward, done, _ = env.step(action)


tensor([[  201.3165, -2291.5437, -3997.0610]], grad_fn=<AddmmBackward0>)
tensor([0])
0
tensor([[  192.5892, -2320.3940, -4041.9448]], grad_fn=<AddmmBackward0>)
tensor([0])
0
tensor([[  203.6746, -2329.8206, -4042.8645]], grad_fn=<AddmmBackward0>)
tensor([0])
0
tensor([[  199.5206, -2327.6033, -4031.5686]], grad_fn=<AddmmBackward0>)
tensor([0])
0
tensor([[  202.2440, -2321.7595, -4037.5232]], grad_fn=<AddmmBackward0>)
tensor([0])
0
tensor([[  204.0302, -2318.7468, -4022.4382]], grad_fn=<AddmmBackward0>)
tensor([0])
0
tensor([[  201.7175, -2320.5842, -4021.6296]], grad_fn=<AddmmBackward0>)
tensor([0])
0
tensor([[  205.4267, -2301.8745, -4021.0698]], grad_fn=<AddmmBackward0>)
tensor([0])
0
tensor([[  201.2245, -2307.9099, -4006.9883]], grad_fn=<AddmmBackward0>)
tensor([0])
0
tensor([[  195.2417, -2317.1987, -4019.1995]], grad_fn=<AddmmBackward0>)
tensor([0])
0
tensor([[  199.8003, -2327.7097, -4026.8677]], grad_fn=<AddmmBackward0>)
tensor([0])
0
tensor([[  199.2415, -2332.1731, -4035.0879

KeyboardInterrupt: 